# 04 — Proposed R-STNet: Spatio-Temporal Graph Consensus & Sensor Reliability Module
**Course:** BCSE209P — Machine Learning Lab  
**Students:** P Naga Venkata Kushal (23BDS0133) & Koteru Tarun (23BDS0103)  
**Faculty Guide:** Dr. Dega Nagaraju  
**Repository:** [Reliability-Aware Anomaly Detection](https://github.com/nvk321/Reliability-Aware-Anomaly-Detection)

---
### Objectives
1. Load the trained PyTorch checkpoint for `R-STNet` (23k parameters, $\tau = 3.1401$).
2. Inspect the learned Graph Attention (GAT) sensor adjacency structure.
3. Examine the three components of the Sensor Reliability Module (SRM):
   - Temporal Continuity ($v_i$)
   - Topological Neighborhood Consistency ($c_i$)
   - Residual Tail Confidence ($p_i$)
4. Demonstrate reliability-weighted anomaly scoring vs. unweighted scoring.


In [ ]:
import os
import sys
import json
import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))
from src.models.rstnet import RSTNet
from src.reliability.sensor_reliability import SensorReliabilityModule
from src.visualization.plots import apply_scientific_style
apply_scientific_style()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")


### 1. Load Trained R-STNet Checkpoint
We load the weights and verify architecture parameters.


In [ ]:
ckpt_path = os.path.join('..', 'results', 'checkpoints', 'proposed_rstnet_swat.pt')
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

model = RSTNet(in_channels=51, hidden_dim=64, node_dim=32, top_k=15).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

tau = float(ckpt['threshold_tau'])
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Loaded R-STNet Checkpoint: {ckpt_path}")
print(f"Calibrated Threshold tau:  {tau:.4f}")
print(f"Trainable Parameters:      {total_params:,}")


### 2. Inspect Learned Sensor Adjacency Graph
R-STNet learns a top-15 sparse directed adjacency matrix between the 51 sensors.


In [ ]:
with torch.no_grad():
    # Extract node embeddings and compute adjacency
    raw_adj = torch.relu(torch.mm(model.graph_attention.node_embed1, model.graph_attention.node_embed2.t()))
    mask = torch.eye(51, device=device).bool()
    raw_adj = raw_adj.masked_fill(mask, 0.0)
    topk_vals, topk_indices = torch.topk(raw_adj, 15, dim=-1)
    sparse_adj = torch.zeros_like(raw_adj)
    sparse_adj.scatter_(-1, topk_indices, topk_vals)
    adj_matrix = torch.softmax(sparse_adj + 1e-6, dim=-1).cpu().numpy()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(adj_matrix, cmap='Blues', interpolation='nearest')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Learned Attention Weight')
ax.set_title('R-STNet Learned Top-15 Sensor Interaction Graph', fontweight='bold')
ax.set_xlabel('Target Sensor Index')
ax.set_ylabel('Source Sensor Index')
plt.tight_layout()
plt.show()


### 3. Forward Pass & Single-Window Diagnostic Evaluation
We run `evaluate_step` on a sample test window to observe the SRM multi-perspective decomposition.


In [ ]:
test_data = np.load(os.path.join('..', 'data', 'processed', 'SWaT', 'test.npy'))
sample_window = test_data[1500:1530]  # Shape (30, 51)

# Restore SRM calibration from checkpoint if available
if 'srm_state' in ckpt and ckpt['srm_state'] is not None:
    model.srm = SensorReliabilityModule.from_dict(ckpt['srm_state'])

diag = model.evaluate_step(sample_window, device=device)

print(f"System Anomaly Score S_sys: {diag['system_score']:.4f} (Threshold tau = {tau:.4f})")
print(f"Most Suspicious Sensor:    Sensor #{diag['most_suspicious_sensor']}")
print(f"Sensor Fault Score:        {diag['sensor_fault_score']:.4f}")
print(f"Mean Reliability:          {diag['reliability_vector'].mean():.4f}")
print(f"Min Reliability:           {diag['reliability_vector'].min():.4f}")


### Viva Voce Key Takeaways — Section D & E: Proposed Model & Reliability
1. **Architecture Formulation:**
   $$\mathbf{X} \xrightarrow{\text{1D TCN}} \mathbf{H}_{\text{temp}} \xrightarrow{\text{GAT}} \mathbf{H}_{\text{joint}} \xrightarrow{\text{Forecast}} \hat{\mathbf{x}}(t)$$
2. **SRM Formulation:**
   $$\text{logit}(r_i) = w_v v_i + w_c c_i + w_p p_i + b$$
3. **Reliability-Weighted Scoring:**
   $$S_{\text{sys}}(t) = \frac{\sum_i r_i(t) z_i(t)}{\sum_i r_i(t) + \epsilon}$$
   When a sensor suffers from severe jitter or drift, its reliability $r_i \to 0$, suppressing its contribution from the system anomaly score and preventing a false plant shutdown.
